In [0]:
%sql
select last_run_status,process_group, * from sandbox.migration_config.table_migration_config where 1=1




In [0]:
%sql
SELECT concat('Truncate TABLE ',target_catalog,'.', target_schema,'.',target_table,';'), *
FROM
  sandbox.migration_config.table_migration_config

In [0]:
%sql
Truncate TABLE sandbox.migration_config_bronze.tpch_region;
Truncate TABLE sandbox.migration_config_bronze.tpcds_promotion;
Truncate TABLE sandbox.migration_config_bronze.customer;
-- Truncate TABLE sandbox.migration_config_bronze.lineitem_copy_into;
Truncate TABLE sandbox.migration_config_bronze.orders;
Truncate TABLE sandbox.migration_config_bronze.tpch_nation;
Truncate TABLE sandbox.migration_config_bronze.tpch_supplier;
Truncate TABLE sandbox.migration_config_bronze.tpch_part;
Truncate TABLE sandbox.migration_config_bronze.tpch_supplier_autoloader;
Truncate TABLE sandbox.migration_config_bronze.tpch_partsupp;
Truncate TABLE sandbox.migration_config_bronze.tpch_lineitem;
Truncate TABLE sandbox.migration_config_bronze.tpch_nation_autoloader;
Truncate TABLE sandbox.migration_config_bronze.pg_neon_supplier_pg;

In [0]:
# Cell 1 — Drop the table (clears all COPY INTO history)
spark.sql("DROP TABLE IF EXISTS sandbox.migration_config_bronze.lineitem_copy_into")
print("Table dropped")

In [0]:
%sql
UPDATE sandbox.migration_config.table_migration_config
 SET
     last_run_status = 'PENDING',
     notes           = NULL,
     last_loaded_value  = NULL,
     last_sf_row_count  = 0,
     last_delta_count  = 0,
     last_run_at = NULL
    --  where last_run_status = 'FAILED'

In [0]:
# Cell 2 — Delete Autoloader checkpoints
# Without this Autoloader thinks the files are already processed
dbutils.fs.rm(
    "/Volumes/sandbox/migration_config_bronze/landing/_checkpoints/TPCH_autoloader/supplier/",
    recurse=True
)
dbutils.fs.rm(
    "/Volumes/sandbox/migration_config_bronze/landing/_checkpoints/TPCH_autoloader/nation/",
    recurse=True
)
print("Checkpoints cleared")

In [0]:
%sql
select last_run_status,process_group,last_run_at, last_delta_count, * from sandbox.migration_config.table_migration_config where 1=1

In [0]:
%sql
SELECT 'tpch_region'              AS table_name, COUNT(*) AS row_count FROM sandbox.migration_config_bronze.tpch_region
UNION ALL
SELECT 'tpcds_promotion',                        COUNT(*) FROM sandbox.migration_config_bronze.tpcds_promotion
UNION ALL
SELECT 'customer',                               COUNT(*) FROM sandbox.migration_config_bronze.customer
UNION ALL
SELECT 'lineitem_copy_into',                     COUNT(*) FROM sandbox.migration_config_bronze.lineitem_copy_into
UNION ALL
SELECT 'orders',                                 COUNT(*) FROM sandbox.migration_config_bronze.orders
UNION ALL
SELECT 'tpch_nation',                            COUNT(*) FROM sandbox.migration_config_bronze.tpch_nation
UNION ALL
SELECT 'tpch_supplier',                          COUNT(*) FROM sandbox.migration_config_bronze.tpch_supplier
UNION ALL
SELECT 'tpch_part',                              COUNT(*) FROM sandbox.migration_config_bronze.tpch_part
UNION ALL
SELECT 'tpch_supplier_autoloader',               COUNT(*) FROM sandbox.migration_config_bronze.tpch_supplier_autoloader
UNION ALL
SELECT 'tpch_partsupp',                          COUNT(*) FROM sandbox.migration_config_bronze.tpch_partsupp
UNION ALL
SELECT 'tpch_lineitem',                          COUNT(*) FROM sandbox.migration_config_bronze.tpch_lineitem
UNION ALL
SELECT 'tpch_nation_autoloader',                 COUNT(*) FROM sandbox.migration_config_bronze.tpch_nation_autoloader
UNION ALL
SELECT 'pg_neon_supplier_pg',                    COUNT(*) FROM sandbox.migration_config_bronze.pg_neon_supplier_pg
ORDER BY table_name;

# Run Migration Jobs in Parallel

This notebook triggers 4 migration framework jobs concurrently and waits for all to complete.

- Jobs could be run independent of each other
- Job could also be scheduled from Workflows
- Job could also be scheduled from ADF

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Define the 4 migration jobs
jobs = [
    {"name": "Snowflake_ForeignCatalog_wf", "job_id": 594266625995112, "process_group": "TPCH_foreign_catalog"},
    {"name": "Autoloader_adls_wf", "job_id": 218903579835428, "process_group": "TPCH_autoloader"},
    {"name": "Postgress_jdbc_Ingestion_wf", "job_id": 1111671871338393, "process_group": "Postgress_jdbc"},
    {"name": "Snowflake_adls_parquet_CopyInto_wf", "job_id": 1007432119292454, "process_group": "VOLUME_copy_into"},
]
# Define the singel migration jobs
# jobs = [
#      {"name": "Postgress_jdbc_Ingestion_wf", "job_id": 1111671871338393, "process_group": "Postgress_jdbc"},
# ]


# Common parameters
common_params = {
    "admin_catalog": "sandbox",
    "config_schema": "migration_config",
    "max_workers": "8",
}

def run_job(job):
    """Trigger a job and wait for completion."""
    try:
        params = {**common_params, "process_group": job["process_group"]}
        print(f"Starting: {job['name']}")
        run = w.jobs.run_now(job_id=job["job_id"], job_parameters=params)
        result = run.result()  # Blocks until complete
        return {"name": job["name"], "status": str(result.state.result_state), "run_id": result.run_id}
    except Exception as e:
        return {"name": job["name"], "status": "FAILED", "error": str(e)}

# Run all jobs in parallel
print("Launching all 4 jobs in parallel...\n")
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(run_job, job): job["name"] for job in jobs}
    results = []
    for future in as_completed(futures):
        result = future.result()
        results.append(result)
        print(f"Completed: {result['name']} -> {result['status']}")

# Summary
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
for r in results:
    status_icon = "✓" if r["status"] == "SUCCESS" else "✗"
    print(f"{status_icon} {r['name']}: {r['status']}")

success_count = sum(1 for r in results if r["status"] == "SUCCESS")
print(f"\n{success_count}/{len(results)} jobs completed successfully")

In [0]:
%sql
select last_run_status,process_group,last_run_at, last_delta_count, * from sandbox.migration_config.table_migration_config where 1=1 ORDER BY target_table;

In [0]:
%sql
SELECT 'tpch_region'              AS table_name, COUNT(*) AS row_count FROM sandbox.migration_config_bronze.tpch_region
UNION ALL
SELECT 'tpcds_promotion',                        COUNT(*) FROM sandbox.migration_config_bronze.tpcds_promotion
UNION ALL
SELECT 'customer',                               COUNT(*) FROM sandbox.migration_config_bronze.customer
UNION ALL
SELECT 'lineitem_copy_into',                     COUNT(*) FROM sandbox.migration_config_bronze.lineitem_copy_into
UNION ALL
SELECT 'orders',                                 COUNT(*) FROM sandbox.migration_config_bronze.orders
UNION ALL
SELECT 'tpch_nation',                            COUNT(*) FROM sandbox.migration_config_bronze.tpch_nation
UNION ALL
SELECT 'tpch_supplier',                          COUNT(*) FROM sandbox.migration_config_bronze.tpch_supplier
UNION ALL
SELECT 'tpch_part',                              COUNT(*) FROM sandbox.migration_config_bronze.tpch_part
UNION ALL
SELECT 'tpch_supplier_autoloader',               COUNT(*) FROM sandbox.migration_config_bronze.tpch_supplier_autoloader
UNION ALL
SELECT 'tpch_partsupp',                          COUNT(*) FROM sandbox.migration_config_bronze.tpch_partsupp
UNION ALL
SELECT 'tpch_lineitem',                          COUNT(*) FROM sandbox.migration_config_bronze.tpch_lineitem
UNION ALL
SELECT 'tpch_nation_autoloader',                 COUNT(*) FROM sandbox.migration_config_bronze.tpch_nation_autoloader
UNION ALL
SELECT 'pg_neon_supplier_pg',                    COUNT(*) FROM sandbox.migration_config_bronze.pg_neon_supplier_pg
ORDER BY table_name;